# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [7]:

EVENT_NAME = '202501_Fire_CA'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'ecostress'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [8]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [9]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 4 .tif files in the S3 bucket.


['drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001_1.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025011030946_aid0001.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025011030946_aid0001_1.tif']

## Configure bucket and paths (no need to create session manually)

In [10]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [11]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 8
  - Total size: 0.01 GB

📁 Cached files (first 10):
  - drcs_activations/202501_Fire_CA/aria/asf/Eaton.tif (4.3 MB)
  - drcs_activations/202501_Fire_CA/aria/asf/Palisades.tif (3.8 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif (0.9 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114.tif (0.4 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112.tif (0.5 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114.tif (0.3 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif (0.1 MB)
  - drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track71_2025-01-09_share.tif (0.1 MB)


(8, 10938970)

In [12]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [13]:
keys

['drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001_1.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025011030946_aid0001.tif',
 'drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025011030946_aid0001_1.tif']

In [22]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Check if it's a control data file
    is_control = 'ControlData' in str(path)
    
    # Extract the doy pattern (e.g., doy2025008040112)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYY-MM-DDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract everything before _doy (includes product, LST, etc.)
        pre_doy = filename.split('_doy')[0]
        
        # Extract everything after the doy pattern (includes aid and any suffix)
        doy_full = f'doy{year}{doy:03d}{time_str}'
        post_doy_parts = filename.split(doy_full)[1]
        
        # Remove leading underscore if present
        if post_doy_parts.startswith('_'):
            post_doy_parts = post_doy_parts[1:]
        
        # Build new filename
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{pre_doy}_{post_doy_parts}_{formatted_datetime}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{pre_doy}_{post_doy_parts}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{filename}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename
filter_str = 'ecostress'

filter_ =  [f for f in keys if filter_str in f]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-08T04:01:12Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-08T04:01:12Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-11T03:09:46Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-11T03:09:46Z.tif


In [23]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/LST", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-08T04:01:12Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-08T04:01:12Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-11T03:09:46Z.tif
  202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-11T03:09:46Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202501_Fire_CA/ecostress
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/LST

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202501_Fire_CA

[1/4] Processing: drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001.tif
   Output filename: 202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-08T04:01:12Z.tif
   [MEMORY] Initial: 306.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection


Reading input: /tmp/tmpiequn85j_temp.tif



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=63797, center sample non-zero=583587/583696
            Estimated data coverage: 99.8% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpj2o5dikv.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-08T04:01:12Z.tif
   [MEMORY] Final: 359.4 MB (Change: +52.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-08T04:01:12Z.tif

[2/4] Processing: drcs_activations/202501_Fire_CA/ecostress/ECO_L2_LSTE.002_LST_doy2025008040112_aid0001_1.tif
   Output filename: 202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-08T04:01:12Z.tif
   [MEMORY] Initial: 359.4 MB
   [DOWNLOAD] Downloading from S3...


...

   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=1275.93994140625, center sample non-zero=583587/583696
            Estimated data coverage: 99.8% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-08T04:01:12Z.tif
   [MEMORY] Final: 346.2 MB (Change: -13.1 MB)
✅ Chun

Reading input: /tmp/tmpcfasuqsb_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp0iahymyc.tif


   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=61807, center sample non-zero=578429/583696
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_2025-01-11T03:09:46Z.tif
   [MEMORY] Final: 360.4 MB (Change: +14.2 MB)
✅ Chunked COG conversion function defined with memory-effi

Reading input: /tmp/tmpq_0zvyi3_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmptchzpv86.tif


   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=1236.1400146484375, center sample non-zero=578429/583696
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202501_Fire_CA_ECO_L2_LSTE.002_LST_aid0001_1_2025-01-11T03:09:46Z.tif
   [MEMORY] Final: 346.2 MB (Change: -14.2 MB)
✅ C

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")